<a href="https://colab.research.google.com/github/bruacosta/python_basic/blob/main/Estudo_Impacto_faturamento_novo_produto.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import os
import numpy as np
import pandas as pd
from scipy import stats

np.random.seed(42)

# ======================================================
# FUNÇÃO PARA GERAR DADOS
# ======================================================

def generatedata(file):

    datas = pd.date_range(
        start='2025-01-01',
        periods=365,
        freq='D'
    )

    # FATURAMENTO BASE
    faturado = np.random.normal(
        loc=2500,
        scale=300,
        size=365
    )

    # NOVO PRODUTO
    VXProductNew = np.zeros(365)

    VXProductNew[199:] = np.random.normal(
        loc=375,
        scale=100,
        size=166
    )

    # CANIBALIZAÇÃO -  considerando que o produto novo vai "roubar" parte das
    # vendas de um produto já disponível, muito comum no varejo

    canibalizacao = 0.75

    vendas_canibalizadas = VXProductNew * canibalizacao

    impacto_incremental = VXProductNew - vendas_canibalizadas

    # FATURAMENTO TOTAL
    FaturamentoTotal = faturado + impacto_incremental

    df = pd.DataFrame({
        'Periodo': datas,
        'FaturamentoBase': faturado,
        'VendasNovoProduto': VXProductNew,
        'ImpactoIncremental': impacto_incremental,
        'FaturamentoTotal': FaturamentoTotal
    })

    return df


# ======================================================
# GERAR OU LER CSV
# ======================================================

file = 'Teste_Hipotese_Impacto_10pct.csv'

if os.path.exists(file):

    print('Arquivo encontrado.')
    df = pd.read_csv(file, sep=';')

else:

    print('Gerando dados...')

    df = generatedata(file)

    df.to_csv(
        file,
        sep=';',
        index=False
    )

    print('Arquivo criado.')


# ======================================================
# TESTE DE HIPÓTESE
# ======================================================

alpha = 0.05

# Antes do produto
antes_produto = df[
    df['VendasNovoProduto'] == 0
]['FaturamentoTotal']

# Depois do produto
depois_produto = df[
    df['VendasNovoProduto'] > 0
]['FaturamentoTotal']

# Média histórica
mu0 = np.mean(antes_produto)

# Impacto mínimo relevante = 10%, tendo em vistas os custo de desenvolvimento
impacto_minimo = mu0 * 0.10

# Nova média mínima aceitável
media_minima_aceitavel = mu0 + impacto_minimo

# Amostra
amostra = depois_produto

# Estatísticas
n = len(amostra)

x_bar = np.mean(amostra)

s = np.std(
    amostra,
    ddof=1
)

ep = s / np.sqrt(n)

# TESTE T UNILATERAL
t_calc, p_value_bilateral = stats.ttest_1samp(
    amostra,
    popmean=media_minima_aceitavel
)

# Convertendo para unilateral
p_value = p_value_bilateral / 2

# t crítico unilateral
t_crit = stats.t.ppf(
    1 - alpha,
    df=n-1
)

# ======================================================
# RESULTADOS
# ======================================================

print('\n========== RESULTADOS ==========')

print(f'Média histórica: {mu0:.2f}')

print(f'Impacto mínimo exigido (10%): {impacto_minimo:.2f}')

print(f'Média mínima aceitável: {media_minima_aceitavel:.2f}')

print(f'Média observada após produto: {x_bar:.2f}')

print(f't calculado: {t_calc:.4f}')

print(f't crítico: {t_crit:.4f}')

print(f'p-value unilateral: {p_value:.6f}')

# ======================================================
# IMPACTO REAL
# ======================================================

impacto_real = x_bar - mu0

impacto_percentual = (impacto_real / mu0) * 100

print('\n========== IMPACTO ==========')

print(f'Impacto médio real: {impacto_real:.2f}')

print(f'Impacto percentual real: {impacto_percentual:.2f}%')

# ======================================================
# DECISÃO FINAL
# ======================================================

print('\n========== DECISÃO ==========')

if p_value < alpha and impacto_percentual >= 10:

    print('REJEITAR H0')
    print('O novo produto gerou impacto estatisticamente')
    print('E o impacto foi relevante para o negócio.')

else:

    print('NÃO REJEITAR H0')
    print('O impacto não atingiu o mínimo relevante de 10%.')

Arquivo encontrado.

========== RESULTADOS ==========
Média histórica: 2489.43
Impacto mínimo exigido (10%): 248.94
Média mínima aceitável: 2738.37
Média observada após produto: 2611.88
t calculado: -5.5936
t crítico: 1.6541
p-value unilateral: 0.000000

========== IMPACTO ==========
Impacto médio real: 122.45
Impacto percentual real: 4.92%

========== DECISÃO ==========
NÃO REJEITAR H0
O impacto não atingiu o mínimo relevante de 10%.


In [9]:
import os
import numpy as np
import pandas as pd
from scipy import stats

np.random.seed(42)

# ======================================================
# FUNÇÃO PARA GERAR DADOS
# ======================================================

def generatedata(file):

    datas = pd.date_range(
        start='2025-01-01',
        periods=365,
        freq='D'
    )

    # Faturamento base
    faturado = np.random.normal(
        loc=2500,
        scale=300,
        size=365
    )

    # Novo produto
    VXProductNew = np.zeros(365)

    # Produto começa no dia 199
    VXProductNew[199:] = np.random.normal(
        loc=375,
        scale=100,
        size=166
    )

    # Canibalização:
    # 75% das vendas do novo produto vêm de produtos antigos
    canibalizacao = 0.75

    impacto_incremental = VXProductNew - (VXProductNew * canibalizacao)

    # Faturamento total considerando impacto líquido
    FaturamentoTotal = faturado + impacto_incremental

    df = pd.DataFrame({
        'Periodo': datas,
        'FaturamentoBase': faturado,
        'VendasNovoProduto': VXProductNew,
        'ImpactoIncremental': impacto_incremental,
        'FaturamentoTotal': FaturamentoTotal
    })

    return df


# ======================================================
# GERAR OU LER ARQUIVO
# ======================================================

file = 'Faturamento_Novo_Produto_Canibalizacao.csv'

if os.path.exists(file):

    print('Arquivo já existe. Carregando CSV...')
    df = pd.read_csv(file, sep=';')

else:

    print('Arquivo não encontrado. Gerando dados...')
    df = generatedata(file)

    df.to_csv(
        file,
        sep=';',
        index=False
    )

    print('Arquivo criado com sucesso.')


# ======================================================
# TESTE DE HIPÓTESE
# ======================================================

# H0: o novo produto NÃO impactou o faturamento médio
# H1: o novo produto impactou o faturamento médio

alpha = 0.05

# Período antes do produto
antes_produto = df[
    df['VendasNovoProduto'] == 0
]['FaturamentoTotal']

# Período após entrada do produto
depois_produto = df[
    df['VendasNovoProduto'] > 0
]['FaturamentoTotal']

# Média histórica antes do produto
mu0 = np.mean(antes_produto)

# Amostra após entrada do produto
amostra = depois_produto

# Tamanho da amostra
n = len(amostra)

# Estatísticas descritivas
x_bar = np.mean(amostra)

s = np.std(
    amostra,
    ddof=1
)

ep = s / np.sqrt(n)

# Teste t de uma amostra
t_calc, p_value = stats.ttest_1samp(
    amostra,
    popmean=mu0
)

# t crítico bilateral
t_crit = stats.t.ppf(
    1 - alpha/2,
    df=n-1
)

# ======================================================
# RESULTADOS
# ======================================================

print('\n========== RESULTADOS ==========')

print(f'Média histórica antes do produto: {mu0:.2f}')
print(f'Média após entrada do produto: {x_bar:.2f}')
print(f'Desvio padrão da amostra: {s:.2f}')
print(f'Erro padrão: {ep:.2f}')
print(f't calculado: {t_calc:.4f}')
print(f't crítico: ±{t_crit:.4f}')
print(f'p-value: {p_value:.6f}')

# ======================================================
# IMPACTO OBSERVADO
# ======================================================

impacto_real = x_bar - mu0
impacto_percentual = (impacto_real / mu0) * 100

print('\n========== IMPACTO OBSERVADO ==========')

print(f'Impacto médio observado: {impacto_real:.2f}')
print(f'Impacto percentual observado: {impacto_percentual:.2f}%')

# ======================================================
# DECISÃO
# ======================================================

print('\n========== DECISÃO ==========')

if p_value < alpha:

    print('REJEITAR H0')
    print('Há evidência estatística de impacto no faturamento.')

else:

    print('NÃO REJEITAR H0')
    print('Não há evidência suficiente de impacto no faturamento.')

Arquivo já existe. Carregando CSV...

========== RESULTADOS ==========
Média histórica antes do produto: 2489.43
Média após entrada do produto: 2611.88
Desvio padrão da amostra: 291.36
Erro padrão: 22.61
t calculado: 5.4146
t crítico: ±1.9744
p-value: 0.000000

========== IMPACTO OBSERVADO ==========
Impacto médio observado: 122.45
Impacto percentual observado: 4.92%

========== DECISÃO ==========
REJEITAR H0
Há evidência estatística de impacto no faturamento.
